# IPS suite — version corrigée

Quatre modèles, un seul socle : **baseline** pleine image, **IPS** mono-échelle,
**MS-IPS** multi-échelle plate, **H-MS-IPS** hiérarchique.

Deux corrections par rapport aux notebooks précédents :

1. **Mémoire bornée.** La boucle en `I` extrait, encode et score les candidats
   par paquets. Le pic est l'image + `M + I` patchs, indépendant de `N`.
   Auparavant les `N` candidats étaient matérialisés et encodés *avant* la boucle.
2. **Résolution native.** Plus de rééchantillonnage vers une taille commune.
   Chaque échelle passe dans l'encodeur à sa taille, l'`AdaptiveAvgPool2d`
   final uniformise la sortie. Supprime le biais qui faisait choisir l'échelle
   restée nette.

À pavage exact, `S` échelles natives coûtent exactement `S` fois une image en
pixels encodés, quelles que soient les tailles de patch.

In [1]:
# ---- CELLULE JETABLE : essai rapide. A SUPPRIMER avant de pousser. ----
import os
os.environ['IPS_RUNS']   = 'ddr_ips,ddr_hmsips_d2'   # ou '' pour tous
os.environ['IPS_EPOCHS'] = '1'
print(">> ESSAI :", os.environ['IPS_RUNS'], "|", os.environ['IPS_EPOCHS'], "epoque")


>> ESSAI : ddr_ips,ddr_hmsips_d2 | 1 epoque


In [2]:
# ============================================================
# Socle : imports, Struct, device, graine
# ============================================================
import os, math, json, time, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, cohen_kappa_score)

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)
    def get(self, k, default=None):
        return self.__dict__.get(k, default)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print(f"PyTorch {torch.__version__} | device {device}")
if torch.cuda.is_available():
    print(f"GPU {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

PyTorch 2.11.0+cu130 | device cuda
GPU Tesla T4 | 15.6 Go


In [3]:
# ============================================================
# Ordonnancement du taux d'apprentissage — identique à IPS original
# ============================================================
def adjust_learning_rate(n_epoch_warmup, n_epoch, max_lr, optimizer, n_step_epoch, step):
    max_steps    = int(n_epoch * n_step_epoch)
    warmup_steps = int(n_epoch_warmup * n_step_epoch)
    if step < warmup_steps:
        lr = max_lr * step / max(warmup_steps, 1)
    else:
        s = step - warmup_steps
        m = max(max_steps - warmup_steps, 1)
        q = 0.5 * (1 + math.cos(math.pi * s / m))
        end_lr = max_lr * 0.001
        lr = max_lr * q + end_lr * (1 - q)
    optimizer.param_groups[0]['lr'] = lr
    return lr

In [4]:
# ============================================================
# Géométrie : pavage exact et extraction vectorisée
#
# Un candidat est un triplet (id_echelle, ligne, colonne). La table des
# candidats ne contient aucun pixel : elle se construit une fois et pèse
# N * 3 entiers. Les pixels ne sont lus que pour les candidats demandés.
# ============================================================
class TileSpec:
    """Pavage exact d'une image carree par plusieurs echelles."""

    def __init__(self, img_size, sizes, device):
        self.img_size = int(img_size)
        self.sizes    = [int(s) for s in sizes]
        rows = []
        for sid, p in enumerate(self.sizes):
            if self.img_size % p != 0:
                raise ValueError(
                    f"img_size={self.img_size} n'est pas divisible par {p}")
            k = self.img_size // p
            r, c = np.divmod(np.arange(k * k), k)
            rows.append(np.stack([np.full(k * k, sid), r * p, c * p], 1))
        cand = np.concatenate(rows, 0).astype(np.int64)
        self.candidates = torch.from_numpy(cand).to(device)   # (N, 3)
        self.n_per_scale = [(self.img_size // p) ** 2 for p in self.sizes]
        self.N = int(self.candidates.shape[0])
        # pixels encodes pour une passe complete : une image par echelle
        self.pixels_per_pass = len(self.sizes) * self.img_size ** 2

    def __repr__(self):
        parts = ', '.join(f"{n}x{p}px" for n, p in
                          zip(self.n_per_scale, self.sizes))
        return (f"TileSpec({self.img_size}^2 : {parts} = {self.N} candidats, "
                f"{self.pixels_per_pass/1e6:.2f} M pixels/passe)")


def extract_patches(img, coords, p):
    """img (C,H,W) -> (n,C,p,p). coords (n,2) = (ligne, colonne).

    Indexation avancee : une seule operation GPU, pas de boucle Python.
    """
    n = coords.shape[0]
    ar = torch.arange(p, device=img.device)
    rr = coords[:, 0].view(n, 1, 1) + ar.view(1, p, 1)
    cc = coords[:, 1].view(n, 1, 1) + ar.view(1, 1, p)
    return img[:, rr, cc].permute(1, 0, 2, 3).contiguous()


def _selftest_geometry():
    spec = TileSpec(896, [128, 64, 32], torch.device('cpu'))
    assert spec.n_per_scale == [49, 196, 784] and spec.N == 1029, spec
    img = torch.rand(3, 896, 896)
    C = spec.candidates
    for sid, p in enumerate(spec.sizes):
        sub = C[C[:, 0] == sid][:7, 1:]
        fast = extract_patches(img, sub, p)
        slow = torch.stack([img[:, int(r):int(r) + p, int(c):int(c) + p]
                            for r, c in sub])
        assert torch.equal(fast, slow), f"extraction {p}px divergente"
    print("  [selftest] geometrie et extraction : OK")

_selftest_geometry()
print(TileSpec(896, [128, 64, 32], torch.device('cpu')))
print(TileSpec(1024, [128, 64], torch.device('cpu')))

  [selftest] geometrie et extraction : OK
TileSpec(896^2 : 49x128px, 196x64px, 784x32px = 1029 candidats, 2.41 M pixels/passe)
TileSpec(1024^2 : 64x128px, 256x64px = 320 candidats, 2.10 M pixels/passe)


In [5]:
# ============================================================
# Encodeur de patch — ResNet-18 tronque, comme dans ips_net.py
#
# n_res_blocks = 4 : coupe apres layer4, sous-echantillonne /32, sortie 512
# n_res_blocks = 2 : coupe apres layer2, sous-echantillonne /8,  sortie 128
#
# A 32 px et 4 blocs la carte de caracteristiques fait 1x1 : l'average
# pooling n'a rien a moyenner. L'original utilise 2 blocs sur MNIST (50 px)
# et 4 sur traffic (100 px).
# ============================================================
class PatchEncoder(nn.Module):

    def __init__(self, n_chan_in=3, pretrained=True, n_res_blocks=4):
        super().__init__()
        if n_res_blocks not in (2, 4):
            raise ValueError("n_res_blocks doit valoir 2 ou 4")
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        net = resnet18(weights=weights)

        if n_chan_in != 3:
            old = net.conv1
            new = nn.Conv2d(n_chan_in, 64, kernel_size=7, stride=2,
                            padding=3, bias=False)
            if pretrained:
                with torch.no_grad():
                    mean_w = old.weight.mean(dim=1, keepdim=True)
                    new.weight.copy_(mean_w.repeat(1, n_chan_in, 1, 1))
            net.conv1 = new

        layers = [net.conv1, net.bn1, net.relu, net.maxpool,
                  net.layer1, net.layer2]
        self.out_dim = 128
        if n_res_blocks == 4:
            layers += [net.layer3, net.layer4]
            self.out_dim = 512
        layers.append(net.avgpool)
        self.net = nn.Sequential(*layers)
        self.downsample = 8 if n_res_blocks == 2 else 32

    def forward(self, x):
        return torch.flatten(self.net(x), 1)

    def feature_map_size(self, patch_size):
        """Cote exact de la carte avant avgpool. 1 signifie degenerescence.

        Verifie a la main : 32->1, 56->2, 64->2, 112->4, 128->4, 224->7,
        256->8 avec 4 blocs. Le simple patch_size//32 est faux (112 donne 3).
        """
        h = (patch_size + 2 * 3 - 7) // 2 + 1      # conv1  k7 s2 p3
        h = (h + 2 * 1 - 3) // 2 + 1               # maxpool k3 s2 p1
        h = (h + 2 * 1 - 3) // 2 + 1               # layer2 s2 (layer1 s1)
        if self.downsample == 32:
            h = (h + 2 * 1 - 3) // 2 + 1           # layer3 s2
            h = (h + 2 * 1 - 3) // 2 + 1           # layer4 s2
        return max(1, h)


class ScaleEmbedding(nn.Module):
    """Un vecteur appris par echelle, ajoute aux caracteristiques projetees."""

    def __init__(self, D, n_scales):
        super().__init__()
        self.emb = nn.Embedding(n_scales, D)
        nn.init.normal_(self.emb.weight, std=0.02)

    def forward(self, feats, scale_ids):
        return feats + self.emb(scale_ids)

In [6]:
# ============================================================
# Transformer a attention croisee — repris de architecture/transformer.py
# (Bergner et al., ICLR 2023). Semantique inchangee.
# ============================================================
class ScaledDotProductAttention(nn.Module):
    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout = nn.Dropout(attn_dropout)

    def compute_attn(self, q, k):
        attn = torch.matmul(q / self.temperature, k.transpose(2, 3))
        return self.dropout(torch.softmax(attn, dim=-1))

    def forward(self, q, k, v):
        return torch.matmul(self.compute_attn(q, k), v)


class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_token, H, D, D_k, D_v, attn_dropout=0.1, dropout=0.1):
        super().__init__()
        self.n_token, self.H, self.D_k, self.D_v = n_token, H, D_k, D_v
        self.q = nn.Parameter(torch.empty((1, n_token, D)))
        v = math.sqrt(1 / D_k)
        nn.init.uniform_(self.q, a=-v, b=v)
        self.q_w = nn.Linear(D, H * D_k, bias=False)
        self.k_w = nn.Linear(D, H * D_k, bias=False)
        self.v_w = nn.Linear(D, H * D_v, bias=False)
        self.fc  = nn.Linear(H * D_v, D, bias=False)
        self.attention  = ScaledDotProductAttention(D_k ** 0.5, attn_dropout)
        self.dropout    = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)

    def get_attn(self, x):
        B, len_seq = x.shape[:2]
        q = self.q_w(self.q).view(1, self.n_token, self.H, self.D_k)
        k = self.k_w(x).view(B, len_seq, self.H, self.D_k)
        return self.attention.compute_attn(q.transpose(1, 2), k.transpose(1, 2))

    def forward(self, x):
        B, len_seq = x.shape[:2]
        q = self.q_w(self.q).view(1, self.n_token, self.H, self.D_k)
        k = self.k_w(x).view(B, len_seq, self.H, self.D_k)
        v = self.v_w(x).view(B, len_seq, self.H, self.D_v)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        x = self.attention(q, k, v)
        x = x.transpose(1, 2).contiguous().view(B, self.n_token, -1)
        x = self.dropout(self.fc(x))
        x = x + self.q
        return self.layer_norm(x)


class MLP(nn.Module):
    def __init__(self, D, D_inner, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(D, D_inner)
        self.w_2 = nn.Linear(D_inner, D)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.dropout(self.w_2(torch.relu(self.w_1(x))))
        return self.layer_norm(x + residual)


class Transformer(nn.Module):
    def __init__(self, n_token, H, D, D_k, D_v, D_inner,
                 attn_dropout=0.1, dropout=0.1):
        super().__init__()
        self.crs_attn = MultiHeadCrossAttention(
            n_token, H, D, D_k, D_v, attn_dropout, dropout)
        self.mlp = MLP(D, D_inner, dropout)

    def get_scores(self, x):
        """Moyenne des scores d'attention sur les tetes puis sur les jetons."""
        attn = self.crs_attn.get_attn(x)
        return attn.mean(dim=1).transpose(1, 2).mean(-1)

    def forward(self, x):
        return self.mlp(self.crs_attn(x))

## Les modèles

`SelectorNet` porte toute la logique commune. `IPS` n'est que le cas `scales`
de longueur 1 ; `MS-IPS` le cas de longueur 2 ou 3 ; `H-MS-IPS` ajoute une
passe grossière avant la passe fine.

La boucle `ips()` ne reçoit jamais de tenseur de patchs : elle reçoit l'image
et une table d'indices, et n'extrait que les `I` candidats du paquet courant.

In [7]:
# ============================================================
# SelectorNet — socle commun IPS / MS-IPS
#
# ips() : exploration sans gradient, par paquets de I candidats.
#   pic memoire = image + (M + I) patchs, independant de N.
# forward() : exploitation avec gradient, sur les M patchs retenus.
#
# Les patchs ne traversent jamais une frontiere de fonction sous forme de
# tenseur empile : on fait circuler des coordonnees (id_echelle, ligne, col).
# ============================================================
class SelectorNet(nn.Module):

    def __init__(self, conf, device):
        super().__init__()
        self.device  = device
        self.conf    = conf
        self.M       = conf.M
        self.I       = conf.I
        self.D       = conf.D
        self.n_class = conf.n_class
        self.shuffle = bool(conf.get('shuffle', True))
        self.scales  = list(conf.scales)

        self.spec = TileSpec(conf.img_size, self.scales, device)

        self.encoder = PatchEncoder(
            n_chan_in=conf.n_chan_in,
            pretrained=conf.pretrained,
            n_res_blocks=conf.n_res_blocks)

        self.projector = nn.Sequential(
            nn.LayerNorm(self.encoder.out_dim, eps=1e-5, elementwise_affine=False),
            nn.Linear(self.encoder.out_dim, conf.D),
            nn.BatchNorm1d(conf.D),
            nn.ReLU())

        self.use_scale_emb = (len(self.scales) > 1
                              and bool(conf.get('use_scale_emb', True)))
        self.scale_emb = (ScaleEmbedding(conf.D, len(self.scales))
                          if self.use_scale_emb else None)

        self.transf = Transformer(
            conf.n_token, conf.H, conf.D, conf.D_k, conf.D_v,
            conf.D_inner, conf.attn_dropout, conf.dropout)

        self.head = nn.Linear(conf.D, conf.n_class)

    # ---------------------------------------------------------------- encodage
    def embed_coords(self, img, coords):
        """(n,3) coordonnees -> (n,D). Un appel d'encodeur par echelle presente.

        L'ordre de sortie suit exactement l'ordre de `coords` : le melange
        aleatoire qui departage les scores ex aequo est donc preserve.
        """
        n = coords.shape[0]
        out = None
        for sid, p in enumerate(self.spec.sizes):
            mask = coords[:, 0] == sid
            if not bool(mask.any()):
                continue
            sub   = coords[mask]
            patch = extract_patches(img, sub[:, 1:], p)
            feats = self.projector(self.encoder(patch))
            if self.scale_emb is not None:
                feats = self.scale_emb(feats, sub[:, 0])
            if out is None:
                out = torch.zeros(n, self.D, device=feats.device,
                                  dtype=feats.dtype)
            out = out.index_put((torch.nonzero(mask, as_tuple=True)[0],), feats)
        if out is None:
            raise RuntimeError("aucun candidat a encoder")
        return out

    # ------------------------------------------------------------- exploration
    @torch.no_grad()
    def select(self, img, candidates, M):
        """Top-M parmi `candidates`, par paquets de I. Renvoie (M,3)."""
        N = int(candidates.shape[0])
        if M >= N:
            return candidates

        was_training = self.training
        if was_training:
            self.encoder.eval(); self.projector.eval(); self.transf.eval()
            if self.scale_emb is not None:
                self.scale_emb.eval()

        cand = candidates
        if self.shuffle:
            cand = cand[torch.randperm(N, device=cand.device)]

        mem_coord = cand[:M]
        mem_emb   = self.embed_coords(img, mem_coord)

        n_iter = math.ceil((N - M) / self.I)
        for i in range(n_iter):
            s = i * self.I + M
            e = min(s + self.I, N)
            it_coord = cand[s:e]
            it_emb   = self.embed_coords(img, it_coord)

            all_emb   = torch.cat([mem_emb, it_emb], dim=0)
            all_coord = torch.cat([mem_coord, it_coord], dim=0)

            scores = self.transf.get_scores(all_emb.unsqueeze(0)).squeeze(0)
            top    = torch.topk(scores, M, dim=-1)[1]

            mem_emb   = all_emb[top]
            mem_coord = all_coord[top]

        if was_training:
            self.encoder.train(); self.projector.train(); self.transf.train()
            if self.scale_emb is not None:
                self.scale_emb.train()
        return mem_coord

    @torch.no_grad()
    def ips(self, img):
        """Point d'entree de la selection pour une image (C,H,W)."""
        return self.select(img, self.spec.candidates, self.M)

    # ---------------------------------------------- tirages de reference
    @torch.no_grad()
    def sample_baseline(self, mode, coords_ref=None):
        """Selection sans apprentissage, pour isoler ce que la selection apporte.

        'uniform'       : M candidats tires uniformement parmi les N de
                          l'image entiere. Aucune passe d'exploration n'est
                          necessaire : c'est la reference la moins chere.
        'scale_matched' : memes comptes par echelle que `coords_ref`, mais
                          positions tirees au hasard dans chaque echelle.
                          Isole le choix des POSITIONS, celui des echelles
                          etant impose.

        L'ecart uniform -> scale_matched mesure ce que vaut le choix des
        echelles ; l'ecart scale_matched -> top-M celui des positions.
        """
        pool = self.spec.candidates
        N = int(pool.shape[0])
        if mode == 'uniform':
            return pool[torch.randperm(N, device=pool.device)[:self.M]]
        if mode == 'scale_matched':
            if coords_ref is None:
                raise ValueError("scale_matched exige coords_ref")
            out = []
            for sid in range(len(self.spec.sizes)):
                k = int((coords_ref[:, 0] == sid).sum())
                if k == 0:
                    continue
                sub = pool[pool[:, 0] == sid]
                pick = torch.randperm(int(sub.shape[0]), device=pool.device)[:k]
                out.append(sub[pick])
            return torch.cat(out, dim=0)
        raise ValueError(f"mode inconnu : {mode}")

    # ------------------------------------------------------------ exploitation
    def classify(self, emb):
        """emb (B,M,D) -> logits (B,n_class)."""
        image_emb = self.transf(emb)          # (B, n_token, D)
        return self.head(image_emb[:, 0])

In [8]:
# ============================================================
# H-MS-IPS — cascade a N etages, du plus grossier au plus fin
#
#   conf.cascade = [224, 56, 14]   tailles, de la plus grossiere a la plus fine
#   conf.M_keep  = [4, 8]          combien garder a chaque etage sauf le dernier
#   conf.M       = 10              budget final
#
# Deux etages = [224, 112] avec M_keep = [4] : l'ancien comportement est un
# cas particulier, rien n'est perdu.
#
# Propriete centrale : les regions retenues a un etage ont une aire fixe, donc
# les subdiviser plus finement multiplie le vivier de l'etage suivant sans
# changer le nombre de pixels encodes.
# ============================================================
class HMSIPSNet(SelectorNet):

    def __init__(self, conf, device):
        cascade = [int(s) for s in conf.cascade]
        M_keep = [int(m) for m in conf.M_keep]
        if len(cascade) < 2:
            raise ValueError("cascade exige au moins deux etages")
        if len(M_keep) != len(cascade) - 1:
            raise ValueError(f"M_keep doit avoir {len(cascade)-1} valeurs, "
                             f"recu {len(M_keep)}")
        for s in cascade:
            if conf.img_size % s != 0:
                raise ValueError(f"img_size {conf.img_size} non divisible par {s}")
        for a, b in zip(cascade, cascade[1:]):
            if a % b != 0 or b >= a:
                raise ValueError(f"etage {a} -> {b} : ratio non entier ou non decroissant")

        inner = Struct(**{**conf.__dict__, 'scales': cascade})
        super().__init__(inner, device)

        self.cascade = cascade
        self.M_keep = M_keep
        self.n_stage = len(cascade)

        # etage 0 : pavage complet de l'image par cascade[0]
        k0 = conf.img_size // cascade[0]
        self.n_zones = k0 * k0
        zr, zc = np.divmod(np.arange(self.n_zones), k0)
        self.zone_coords = torch.from_numpy(
            np.stack([np.zeros(self.n_zones, np.int64),
                      zr * cascade[0], zc * cascade[0]], 1)).to(device)

        # decalages d'un parent vers ses enfants, un jeu par transition
        self.offsets = []
        for a, b in zip(cascade, cascade[1:]):
            r = a // b
            fr, fc = np.divmod(np.arange(r * r), r)
            self.offsets.append(torch.from_numpy(
                np.stack([fr * b, fc * b], 1).astype(np.int64)).to(device))

        # tailles de vivier, pour le journal
        self.pool_sizes = [self.n_zones]
        for k, off in enumerate(self.offsets):
            self.pool_sizes.append(M_keep[k] * int(off.shape[0]))

    def _subdivide(self, parents, stage):
        """parents (k,3) de l'etage `stage` -> enfants (k*r^2, 3) a l'etage+1."""
        off = self.offsets[stage]
        r2 = int(off.shape[0])
        base = parents[:, 1:].repeat_interleave(r2, dim=0)
        shift = off.repeat(int(parents.shape[0]), 1)
        sid = torch.full((base.shape[0], 1), stage + 1,
                         dtype=torch.int64, device=base.device)
        return torch.cat([sid, base + shift], dim=1)

    @torch.no_grad()
    def ips(self, img):
        keep = self.select(img, self.zone_coords, self.M_keep[0])
        for stage in range(self.n_stage - 1):
            child = self._subdivide(keep, stage)
            budget = (self.M if stage == self.n_stage - 2
                      else self.M_keep[stage + 1])
            keep = self.select(img, child, budget)
        return keep

    @property
    def pixels_per_pass(self):
        """Image entiere a l'etage 0, puis les regions retenues aux suivants."""
        px = self.conf.img_size ** 2
        for k, m in enumerate(self.M_keep):
            px += m * self.cascade[k] ** 2
        return px

    @property
    def n_fine_cand(self):
        return self.pool_sizes[-1]

In [9]:
# ============================================================
# Baseline — ResNet-18 de bout en bout sur l'image sous-echantillonnee
# Meme dorsale que les modeles a patchs : seule la selection change.
# ============================================================
class BaselineNet(nn.Module):

    def __init__(self, conf, device):
        super().__init__()
        self.device = device
        self.conf   = conf
        self.input_size = int(conf.get('baseline_size', conf.img_size))

        weights = ResNet18_Weights.IMAGENET1K_V1 if conf.pretrained else None
        net = resnet18(weights=weights)
        if conf.n_chan_in != 3:
            old = net.conv1
            new = nn.Conv2d(conf.n_chan_in, 64, 7, 2, 3, bias=False)
            with torch.no_grad():
                new.weight.copy_(
                    old.weight.mean(dim=1, keepdim=True).repeat(1, conf.n_chan_in, 1, 1))
            net.conv1 = new
        net.fc = nn.Linear(512, conf.n_class)
        self.net = net

    def forward(self, imgs):
        if imgs.shape[-1] != self.input_size:
            imgs = F.interpolate(imgs, size=(self.input_size, self.input_size),
                                 mode='bilinear', align_corners=False)
        return self.net(imgs)

In [10]:
# ============================================================
# Exploitation par lot — un appel d'encodeur par echelle,
# UN SEUL appel de projecteur pour tout le lot.
#
# Le projecteur contient un BatchNorm1d : le decouper par echelle changerait
# ses statistiques de lot. On concatene donc les sorties brutes de l'encodeur
# avant de projeter, ce qui reproduit exactement l'ancien comportement.
# ============================================================
def _embed_from_store(net, store, B):
    """store[sid] = (liste de patchs, liste d'indices lot, liste d'indices slot)."""
    raws, bidx, sidx, sids = [], [], [], []
    for sid in range(len(net.spec.sizes)):
        pats, bb, ss = store[sid]
        if not pats:
            continue
        x = torch.cat(pats, dim=0)
        raws.append(net.encoder(x))
        bidx.append(torch.cat(bb))
        sidx.append(torch.cat(ss))
        sids.append(torch.full((x.shape[0],), sid, dtype=torch.long,
                               device=x.device))
    if not raws:
        raise RuntimeError("lot vide")
    feats = net.projector(torch.cat(raws, dim=0))
    if net.scale_emb is not None:
        feats = net.scale_emb(feats, torch.cat(sids))
    emb = torch.zeros(B, net.M, net.D, device=feats.device, dtype=feats.dtype)
    return emb.index_put((torch.cat(bidx), torch.cat(sidx)), feats)


def _new_store(n_scales):
    return {sid: ([], [], []) for sid in range(n_scales)}


@torch.no_grad()
def _fill_store(net, store, img, coords, b):
    """Extrait les M patchs retenus et les range par echelle. Sans gradient :
    ce sont des donnees, le graphe commence a l'encodeur."""
    for sid, p in enumerate(net.spec.sizes):
        mask = coords[:, 0] == sid
        if not bool(mask.any()):
            continue
        idx = torch.nonzero(mask, as_tuple=True)[0]
        sub = coords[idx]
        pats, bb, ss = store[sid]
        pats.append(extract_patches(img, sub[:, 1:], p))
        bb.append(torch.full((idx.numel(),), b, dtype=torch.long,
                             device=idx.device))
        ss.append(idx)

In [11]:
# ============================================================
# Metriques — binaire (RSNA) ou ordinale 5 classes (DDR)
# ============================================================
def compute_metrics(y_true, y_pred, y_prob, task_type):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    out = {'accuracy': float(accuracy_score(y_true, y_pred))}
    if task_type == 'ordinal':
        out['kappa'] = float(cohen_kappa_score(y_true, y_pred,
                                               weights='quadratic'))
        out['f1'] = float(f1_score(y_true, y_pred, average='macro',
                                   zero_division=0))
    else:
        out['f1'] = float(f1_score(y_true, y_pred, zero_division=0))
        out['precision'] = float(precision_score(y_true, y_pred,
                                                 zero_division=0))
        out['recall'] = float(recall_score(y_true, y_pred, zero_division=0))
        try:
            out['auc'] = float(roc_auc_score(y_true, np.asarray(y_prob)[:, 1]))
        except ValueError:
            out['auc'] = float('nan')
    return out

In [12]:
# ============================================================
# Perte — reproduit exactement celle d'IPS original :
# softmax, puis log(p + eps), puis NLLLoss. Le +eps n'est pas cosmetique,
# il change la valeur ; on le garde pour pouvoir comparer aux runs passes.
# ============================================================
class IPSLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.nll = nn.NLLLoss()

    def forward(self, logits, target):
        p = torch.softmax(logits, dim=-1)
        return self.nll(torch.log(p + self.eps), target)

In [13]:
# ============================================================
# Une epoque
#
# Le pas d'apprentissage est compte en iterations du DataLoader, comme dans
# training/iterative.py : max_steps = n_epoch * len(loader), avec un loader
# de taille de lot B_seq. L'optimiseur, lui, avance tous les B images.
# ============================================================
def run_epoch(net, loader, conf, device, optimizer=None, epoch=0,
              criterion=None, is_baseline=False):
    train = optimizer is not None
    net.train(train)

    tot_loss, n_seen = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    n_step_epoch = max(1, len(loader))
    it = epoch * n_step_epoch

    if is_baseline:
        with (torch.enable_grad() if train else torch.no_grad()):
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                it += 1
                if train:
                    adjust_learning_rate(conf.n_epoch_warmup, conf.n_epoch,
                                         conf.lr, optimizer, n_step_epoch, it)
                    optimizer.zero_grad()
                logits = net(imgs)
                loss = criterion(logits, labels)
                if train:
                    loss.backward()
                    optimizer.step()
                bs = labels.shape[0]
                tot_loss += loss.item() * bs
                n_seen += bs
                prob = torch.softmax(logits.detach(), -1)
                y_true += labels.tolist()
                y_pred += prob.argmax(-1).tolist()
                y_prob += prob.cpu().numpy().tolist()
        return tot_loss / max(n_seen, 1), y_true, y_pred, y_prob

    store = _new_store(len(net.spec.sizes))
    labels_buf, n_prep = [], 0

    def flush(step_it):
        nonlocal store, labels_buf, n_prep, tot_loss, n_seen
        if n_prep == 0:
            return
        B = n_prep
        lab = torch.stack(labels_buf)
        if train:
            adjust_learning_rate(conf.n_epoch_warmup, conf.n_epoch,
                                 conf.lr, optimizer, n_step_epoch, step_it)
            optimizer.zero_grad()
        with (torch.enable_grad() if train else torch.no_grad()):
            emb = _embed_from_store(net, store, B)
            logits = net.classify(emb)
            loss = criterion(logits, lab)
            if train:
                loss.backward()
                optimizer.step()
        tot_loss += loss.item() * B
        n_seen += B
        prob = torch.softmax(logits.detach(), -1)
        y_true.extend(lab.tolist())
        y_pred.extend(prob.argmax(-1).tolist())
        y_prob.extend(prob.cpu().numpy().tolist())
        store = _new_store(len(net.spec.sizes))
        labels_buf = []
        n_prep = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        it += 1
        for k in range(imgs.shape[0]):
            coords = net.ips(imgs[k])
            _fill_store(net, store, imgs[k], coords, n_prep)
            labels_buf.append(labels[k])
            n_prep += 1
            if n_prep == conf.B:
                flush(it)
    flush(it)
    return tot_loss / max(n_seen, 1), y_true, y_pred, y_prob

In [14]:
# ============================================================
# Jeux de donnees — reprises telles quelles des notebooks precedents,
# simplifiees pour renvoyer (image, label) au lieu d'un dictionnaire.
# ============================================================
class DDRDataset(Dataset):
    """DDR, 5 grades (0-4). Le grade 5 (non gradable) est exclu."""

    def __init__(self, conf, split):
        assert split in ('train', 'valid', 'test')
        self.img_size = conf.img_size
        root = Path(conf.data_dir) / 'DR_grading'
        lab_file, img_dir = root / f'{split}.txt', root / split

        self._data = []
        with open(lab_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                fname, grade = line.split()[0], int(line.split()[1])
                if grade == 5:
                    continue
                p = img_dir / fname
                if p.exists():
                    self._data.append((p, grade))

        counts = [0] * 5
        for _, g in self._data:
            counts[g] += 1
        print(f"[DDR] {split:5s} | {len(self._data):5d} images | "
              + ', '.join(f"C{i}={counts[i]}" for i in range(5)))

        norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])
        if split == 'train':
            self.tf = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                       saturation=0.1, hue=0.05),
                norm])
        else:
            self.tf = norm

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        path, label = self._data[i]
        img = Image.open(path).convert('RGB').resize(
            (self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2, 0, 1)
        return self.tf(t), label


class RSNADataset(Dataset):
    """RSNA pneumonie, binaire. Split 75/25 par hachage MD5 du patientId."""

    @staticmethod
    def _to_split(pid, seed=42):
        h = int(hashlib.md5((str(pid) + str(seed)).encode()).hexdigest(), 16)
        return 'test' if (h % 100) < 25 else 'train'

    def __init__(self, conf, split):
        assert split in ('train', 'test')
        self.img_size = conf.img_size
        root = Path(conf.data_dir)
        img_dir = root / 'Training' / 'Images'

        df = (pd.read_csv(root / 'stage2_train_metadata.csv')
              .groupby('patientId')['Target'].max().reset_index())
        df['split'] = df['patientId'].apply(
            lambda p: self._to_split(p, seed=conf.seed))
        df = df[df['split'] == split].reset_index(drop=True)

        self._data, missing = [], 0
        for _, row in df.iterrows():
            p = img_dir / f"{row['patientId']}.png"
            if p.exists():
                self._data.append((p, int(row['Target'])))
            else:
                missing += 1
        pos = sum(l for _, l in self._data)
        print(f"[RSNA] {split:5s} | {len(self._data):5d} images | "
              f"pos={pos} ({100*pos/max(len(self._data),1):.1f}%)"
              + (f" | {missing} manquantes" if missing else ""))

        self.aug = (transforms.Compose([transforms.RandomHorizontalFlip(),
                                        transforms.RandomRotation(5)])
                    if split == 'train' else None)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        path, label = self._data[i]
        img = Image.open(path).convert('L')
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        t = (torch.from_numpy(arr).unsqueeze(0) - 0.5) / 0.5
        if self.aug is not None:
            t = self.aug(t)
        return t, label


def build_loaders(conf):
    if conf.dataset == 'ddr':
        tr = DDRDataset(conf, 'train')
        va = DDRDataset(conf, 'valid')
    elif conf.dataset == 'rsna':
        tr = RSNADataset(conf, 'train')
        va = RSNADataset(conf, 'test')
    else:
        raise ValueError(conf.dataset)
    # La baseline fait un pas par lot du DataLoader ; les modeles a patchs
    # accumulent B_seq images jusqu'a B avant de faire le leur. Pour que les
    # deux s'entrainent avec le MEME lot de B=16, la baseline charge par B.
    bs = conf.B if conf.model == 'baseline' else conf.B_seq
    kw = dict(num_workers=conf.n_worker, pin_memory=conf.pin_memory,
              persistent_workers=conf.n_worker > 0)
    return (DataLoader(tr, batch_size=bs, shuffle=True, **kw),
            DataLoader(va, batch_size=bs, shuffle=False, **kw))

## Configurations

Une seule base par jeu de données, déclinée par modèle. Tout ce qui n'est pas
nommé dans la déclinaison est hérité, donc la dorsale, l'optimiseur et la
graine sont identiques partout : la seule variable est la sélection.

`n_res_blocks = 2` coupe le ResNet après `layer2`. À 32 px, 4 blocs donnent une
carte 1×1 ; 2 blocs donnent 4×4.

In [15]:
# ============================================================
# Chemins — pilotes par variables d'environnement.
#
# Sur Kaggle aucune variable n'est posee : les valeurs par defaut
# s'appliquent. Sur le cluster, run-ips-suite.sh les exporte. Il n'y a donc
# aucune reecriture de chemins dans le script, donc rien qui puisse echouer
# silencieusement sur une chaine mal appariee.
# ============================================================
def _env_path(key, default):
    return Path(os.environ.get(key, str(default)))

WORK     = _env_path('IPS_WORK', '/kaggle/working')
OUT_CKPT = _env_path('IPS_OUT_CKPT', WORK / 'checkpoints')
OUT_HIST = _env_path('IPS_OUT_HIST', WORK / 'results')
OUT_EXPE = _env_path('IPS_OUT_EXPE', WORK / 'experiments')
OUT_SAL  = _env_path('IPS_OUT_SAL',  WORK / 'saliency')
for d in (OUT_CKPT, OUT_HIST, OUT_EXPE, OUT_SAL):
    d.mkdir(parents=True, exist_ok=True)

DDR_DIR = os.environ.get(
    'IPS_DDR_DIR',
    '/kaggle/input/datasets/samriddhibagchi/ddr-dataset-credits-to-authors/DDR-dataset')
RSNA_DIR = os.environ.get(
    'IPS_RSNA_DIR',
    '/kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset')

print(f"  travail     : {WORK}")
print(f"  checkpoints : {OUT_CKPT}")
print(f"  DDR         : {DDR_DIR}")
print(f"  RSNA        : {RSNA_DIR}")

BASE = dict(
    n_epoch=50, n_epoch_warmup=10, lr=0.0001, wd=0.1,
    B=16, B_seq=4, seed=42, eps=1e-6,
    M=10, I=10, D=512, H=8, D_k=64, D_v=64, D_inner=2048,
    n_token=1, attn_dropout=0.1, dropout=0.1,
    pretrained=True, n_res_blocks=4, shuffle=True, use_scale_emb=True,
    n_worker=2, pin_memory=True,
)

DDR_BASE = dict(BASE, dataset='ddr', data_dir=DDR_DIR, img_size=896,
                n_chan_in=3, n_class=5, task_type='ordinal',
                main_metric='kappa')

RSNA_BASE = dict(BASE, dataset='rsna', data_dir=RSNA_DIR, img_size=1024,
                 n_chan_in=1, n_class=2, task_type='binary',
                 main_metric='auc')


def cfg(base, name, model, **kw):
    d = dict(base, run_name=name, model=model, **kw)
    # Un modele hierarchique est decrit par `cascade` ; on en derive `scales`
    # pour que tout le reste du notebook (journal, affichages) lise une seule
    # et meme cle, quel que soit le type de modele.
    if model == 'hierarchical' and 'scales' not in d:
        d['scales'] = list(d['cascade'])
    return Struct(**d)


CONFIGS = {
    # ---- DDR ----------------------------------------------------------
    'ddr_baseline': cfg(DDR_BASE, 'ddr_baseline_v2', 'baseline',
                        baseline_size=896, scales=[128]),
    'ddr_ips':      cfg(DDR_BASE, 'ddr_ips_v2', 'selector', scales=[128]),
    'ddr_msips2':   cfg(DDR_BASE, 'ddr_msips2_native_v2', 'selector',
                        scales=[128, 64]),
    'ddr_msips3':   cfg(DDR_BASE, 'ddr_msips3_native_v2', 'selector',
                        scales=[128, 64, 32], n_res_blocks=2),
    # profondeur d'encodeur : variable de controle, pas reglage discret
    'ddr_ips_b2':   cfg(DDR_BASE, 'ddr_ips_b2_v2', 'selector',
                        scales=[128], n_res_blocks=2),
    'ddr_msips2_b2': cfg(DDR_BASE, 'ddr_msips2_native_b2_v2', 'selector',
                         scales=[128, 64], n_res_blocks=2),
    # subdivision des zones retenues : meme nombre de pixels, vivier x4 puis x16
    # ---- H-MS-IPS a deux etages, subdivision croissante -----------------
    'ddr_hmsips_d2': cfg(DDR_BASE, 'ddr_hmsips_d2_v2', 'hierarchical',
                         cascade=[224, 112], M_keep=[4]),
    'ddr_hmsips_d4': cfg(DDR_BASE, 'ddr_hmsips_d4_v2', 'hierarchical',
                         cascade=[224, 56], M_keep=[4]),
    'ddr_hmsips_d8': cfg(DDR_BASE, 'ddr_hmsips_d8_v2', 'hierarchical',
                         cascade=[224, 28], M_keep=[4], n_res_blocks=2),
    # fin = 14 px = 204 um, soit 1,6x le plafond ETDRS du micro-anevrisme
    'ddr_hmsips_d16': cfg(DDR_BASE, 'ddr_hmsips_d16_v2', 'hierarchical',
                          cascade=[224, 14], M_keep=[4], n_res_blocks=2,
                          I=128),
    # ---- H-MS-IPS a TROIS etages ----------------------------------------
    # 224 -> 56 -> 14 um : chaque etage divise la taille par 4, donc le
    # vivier par 16. Elagage reel a chaque passage : 4/16, 8/64, 10/128.
    'ddr_hmsips3': cfg(DDR_BASE, 'ddr_hmsips3_v2', 'hierarchical',
                       cascade=[224, 56, 14], M_keep=[4, 8],
                       n_res_blocks=2),
    # echelle clinique plate : contexte 128 (1,24 papille), amas 32 (466 um),
    # micro-anevrisme 16 (233 um, 1,9x le plafond)
    'ddr_msips3_clin': cfg(DDR_BASE, 'ddr_msips3_clinique_v2', 'selector',
                           scales=[128, 32, 16], n_res_blocks=2, I=128),
    # ---- RSNA ---------------------------------------------------------
    'rsna_baseline': cfg(RSNA_BASE, 'rsna_baseline_v2', 'baseline',
                         baseline_size=1024, scales=[128]),
    'rsna_ips':      cfg(RSNA_BASE, 'rsna_ips_v2', 'selector', scales=[128]),
    'rsna_msips2':   cfg(RSNA_BASE, 'rsna_msips2_native_v2', 'selector',
                         scales=[128, 64]),
    'rsna_msips3':   cfg(RSNA_BASE, 'rsna_msips3_native_v2', 'selector',
                         scales=[128, 64, 32], n_res_blocks=2),
    # echelle clinique RSNA : l'opacite fait ~215 px, donc 128 est DEJA
    # sub-lesionnel. La bonne echelle est plus grossiere, pas plus fine.
    'rsna_msips2_clin': cfg(RSNA_BASE, 'rsna_msips2_clinique_v2', 'selector',
                            scales=[256, 128]),
    'rsna_hmsips_d2': cfg(RSNA_BASE, 'rsna_hmsips_d2_v2', 'hierarchical',
                          cascade=[256, 128], M_keep=[4]),
    'rsna_hmsips_d4': cfg(RSNA_BASE, 'rsna_hmsips_d4_v2', 'hierarchical',
                          cascade=[256, 64], M_keep=[4]),
    'rsna_hmsips_d8': cfg(RSNA_BASE, 'rsna_hmsips_d8_v2', 'hierarchical',
                          cascade=[256, 32], M_keep=[4], n_res_blocks=2),
    # trois etages sur RSNA : controle. L'argument clinique dit que l'etage
    # le plus fin (64 px = 22 mm pour une lesion de 75 mm) n'apportera rien.
    'rsna_hmsips3': cfg(RSNA_BASE, 'rsna_hmsips3_v2', 'hierarchical',
                        cascade=[256, 128, 64], M_keep=[4, 6]),
}

for k, c in CONFIGS.items():
    extra = (f"  M_keep={c.M_keep}" if c.model == 'hierarchical' else '')
    print(f"  {k:17s} {c.run_name:26s} {c.model:12s} "
          f"scales={c.scales}{extra}")

  travail     : /kaggle/working
  checkpoints : /kaggle/working/checkpoints
  DDR         : /kaggle/input/datasets/samriddhibagchi/ddr-dataset-credits-to-authors/DDR-dataset
  RSNA        : /kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset
  ddr_baseline      ddr_baseline_v2            baseline     scales=[128]
  ddr_ips           ddr_ips_v2                 selector     scales=[128]
  ddr_msips2        ddr_msips2_native_v2       selector     scales=[128, 64]
  ddr_msips3        ddr_msips3_native_v2       selector     scales=[128, 64, 32]
  ddr_ips_b2        ddr_ips_b2_v2              selector     scales=[128]
  ddr_msips2_b2     ddr_msips2_native_b2_v2    selector     scales=[128, 64]
  ddr_hmsips_d2     ddr_hmsips_d2_v2           hierarchical scales=[224, 112]  M_keep=[4]
  ddr_hmsips_d4     ddr_hmsips_d4_v2           hierarchical scales=[224, 56]  M_keep=[4]
  ddr_hmsips_d8     ddr_hmsips_d8_v2           hierarchical scales=[224, 28]  M_keep=[4]
  ddr_hmsips_d16    d

In [16]:
# ============================================================
# Construction du modele + budget en pixels encodes, affiche avant le run
# ============================================================
def build_model(conf, device):
    if conf.model == 'baseline':
        return BaselineNet(conf, device).to(device)
    if conf.model == 'hierarchical':
        return HMSIPSNet(conf, device).to(device)
    if conf.model == 'selector':
        return SelectorNet(conf, device).to(device)
    raise ValueError(conf.model)


def cost_report(conf, net):
    """Pixels encodes par image pour une passe d'exploration complete."""
    hw = conf.img_size ** 2
    if conf.model == 'baseline':
        px, n_cand = conf.get('baseline_size', conf.img_size) ** 2, 0
    elif conf.model == 'hierarchical':
        px = net.pixels_per_pass
        n_cand = net.n_zones + net.n_fine_cand
    else:
        px = net.spec.pixels_per_pass
        n_cand = net.spec.N
    lines = [f"  candidats          : {n_cand}",
             f"  pixels encodes     : {px/1e6:.2f} M  ({px/hw:.2f} x H*W)"]
    if conf.model != 'baseline':
        enc = net.encoder
        maps = ', '.join(f"{p}px->{enc.feature_map_size(p)}x{enc.feature_map_size(p)}"
                         for p in net.spec.sizes)
        lines.append(f"  cartes             : {maps}")
        if any(enc.feature_map_size(p) == 1 for p in net.spec.sizes):
            lines.append("  [ATTENTION] une carte est 1x1 : passer n_res_blocks a 2")
        peak = (conf.M + conf.I) * conf.n_chan_in * max(net.spec.sizes) ** 2 * 4
        lines.append(f"  pic patchs (M+I)   : {peak/1e6:.1f} Mo + image "
                     f"{conf.n_chan_in*hw*4/1e6:.1f} Mo")
    print('\n'.join(lines))
    return px, n_cand

In [17]:
# ============================================================
# Journalisation — meme format que les experiences precedentes
# ============================================================
def log_experiment(dir_path, conf, net, history, timing, px, n_cand):
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    csv_path = Path(dir_path) / f"{conf.run_name}_experiment.csv"

    def best(metric, mode='max'):
        vals = [h['val'].get(metric) for h in history]
        vals = [(v, i) for i, v in enumerate(vals)
                if v is not None and not (isinstance(v, float) and math.isnan(v))]
        if not vals:
            return float('nan'), -1
        v, i = (max(vals) if mode == 'max' else min(vals))
        return v, i + 1

    row = {
        'run_name': conf.run_name,
        'date': time.strftime('%Y-%m-%d %H:%M'),
        'dataset': conf.dataset, 'model': conf.model,
        'n_epochs': conf.n_epoch, 'lr': conf.lr, 'wd': conf.wd,
        'B': conf.B, 'B_seq': conf.B_seq, 'M': conf.M, 'I': conf.I,
        'D': conf.D, 'H': conf.H, 'n_epoch_warmup': conf.n_epoch_warmup,
        'enc_type': 'resnet18', 'n_res_blocks': conf.n_res_blocks,
        'pretrained': conf.pretrained,
        'img_size': conf.img_size,
        'scales': '-'.join(str(s) for s in conf.scales),
        'resample': 'native',
        'n_candidates': n_cand,
        'pixels_per_pass_M': round(px / 1e6, 3),
    }
    bl, be = best('loss', 'min')
    row['best_val_loss'], row['best_ep_loss'] = bl, be
    for m in ('accuracy', 'kappa', 'f1', 'auc', 'precision', 'recall'):
        v, e = best(m, 'max')
        if not math.isnan(v):
            row[f'best_val_{m}'], row[f'best_ep_{m}'] = v, e
    last = history[-1]
    row['final_train_loss'] = last['train']['loss']
    row['final_val_loss'] = last['val']['loss']
    for m, v in last['train'].items():
        if m != 'loss':
            row[f'final_train_{m}'] = v
    for m, v in last['val'].items():
        if m != 'loss':
            row[f'final_val_{m}'] = v
    row.update(timing)

    with open(csv_path, 'w') as f:
        f.write('### RESUME\n')
        f.write(','.join(row.keys()) + '\n')
        f.write(','.join(str(v) for v in row.values()) + '\n')
        f.write('\n### HISTORIQUE\n')
        cols = ['epoch', 'train_loss', 'val_loss'] + [
            f'{s}_{m}' for s in ('train', 'val')
            for m in last['train'] if m != 'loss']
        f.write(','.join(cols) + '\n')
        for h in history:
            vals = [h['epoch'], h['train']['loss'], h['val']['loss']]
            for s in ('train', 'val'):
                for m in last['train']:
                    if m != 'loss':
                        vals.append(h[s].get(m, ''))
            f.write(','.join(str(v) for v in vals) + '\n')
    print(f"  CSV : {csv_path}")
    return csv_path

In [18]:
# ============================================================
# Saillance — exporte les coordonnees retenues et une vignette par image
# ============================================================
@torch.no_grad()
def save_saliency(net, dataset, conf, save_dir, n_images=5, seed=42):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), min(n_images, len(dataset)))
    net.eval()
    colors = ['#d62728', '#1f77b4', '#2ca02c', '#9467bd']
    rows = []

    for k, i in enumerate(idxs):
        img, label = dataset[i]
        img = img.to(net.device)
        coords_t = net.ips(img)
        coords = coords_t.cpu().numpy()
        logits = net.classify(_embed_single(net, img, coords_t))
        prob = torch.softmax(logits, -1)[0].cpu().numpy()
        pred = int(prob.argmax())

        disp = img.cpu().numpy()
        disp = disp.mean(0) if disp.shape[0] == 1 else disp.transpose(1, 2, 0)
        disp = (disp - disp.min()) / max(float(disp.max() - disp.min()), 1e-8)

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(disp, cmap='gray' if disp.ndim == 2 else None)
        for sid, r, c in coords:
            p = net.spec.sizes[sid]
            ax.add_patch(mpatches.Rectangle(
                (c, r), p, p, fill=False, lw=2.0,
                edgecolor=colors[sid % len(colors)]))
        ax.set_title(f"{conf.run_name} | vrai={label} predit={pred} "
                     f"p={prob[pred]:.3f}")
        ax.axis('off')
        fig.tight_layout()
        fig.savefig(save_dir / f"{conf.run_name}_saliency_{k:02d}_img{i}.png",
                    dpi=110)
        plt.close(fig)

        for rank, (sid, r, c) in enumerate(coords):
            rows.append(dict(img_index=i, true=int(label), pred=pred,
                             prob=float(prob[pred]), patch_rank=rank,
                             scale_px=net.spec.sizes[sid],
                             row=int(r), col=int(c)))

    df = pd.DataFrame(rows)
    df.to_csv(save_dir / f"{conf.run_name}_saliency.csv", index=False)
    counts = df.scale_px.value_counts().sort_index()
    print("  repartition des echelles sur ces images : "
          + ', '.join(f"{p}px {n}/{len(df)}" for p, n in counts.items()))
    return df


def _embed_single(net, img, coords):
    store = _new_store(len(net.spec.sizes))
    _fill_store(net, store, img, coords, 0)
    return _embed_from_store(net, store, 1)


@torch.no_grad()
def scale_distribution(net, loader, device, max_images=None):
    """Repartition des echelles sur TOUT le jeu, pas sur cinq images."""
    net.eval()
    counts = np.zeros(len(net.spec.sizes), dtype=np.int64)
    seen = 0
    for imgs, _ in loader:
        imgs = imgs.to(device)
        for k in range(imgs.shape[0]):
            c = net.ips(imgs[k])[:, 0].cpu().numpy()
            counts += np.bincount(c, minlength=len(net.spec.sizes))
            seen += 1
            if max_images is not None and seen >= max_images:
                break
        if max_images is not None and seen >= max_images:
            break
    tot = int(counts.sum())
    print(f"  echelles sur {seen} images : " + ', '.join(
        f"{p}px {int(n)}/{tot} ({100*n/max(tot,1):.1f}%)"
        for p, n in zip(net.spec.sizes, counts)))
    return dict(zip(net.spec.sizes, counts.tolist())), seen

In [19]:
# ============================================================
# Evaluation des strategies de selection — repond a la note de reunion
# « uniform sampling (baseline) and sampling following the real proportion ».
#
#   top-M          : la selection apprise
#   scale_matched  : memes comptes par echelle, positions au hasard
#   uniform        : tout au hasard parmi les N candidats
#
# La decomposition : uniform -> scale_matched dit ce que vaut le choix des
# echelles, scale_matched -> top-M ce que vaut le choix des positions.
#
# Cout : uniform ne demande AUCUNE passe d'exploration. scale_matched en
# demande une, puisqu'il faut les comptes du modele. C'est rapporte tel quel.
# ============================================================
@torch.no_grad()
def evaluate_selection(net, loader, conf, device, mode, criterion):
    """mode : 'topm' | 'scale_matched' | 'uniform'. Renvoie (metriques, couts)."""
    net.eval()
    store = _new_store(len(net.spec.sizes))
    labels_buf, n_prep = [], 0
    tot_loss, n_seen = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    px_explore, px_classify, n_img = 0, 0, 0
    scale_counts = np.zeros(len(net.spec.sizes), dtype=np.int64)

    explore_px_img = (net.pixels_per_pass if conf.model == 'hierarchical'
                      else net.spec.pixels_per_pass)

    def flush():
        nonlocal store, labels_buf, n_prep, tot_loss, n_seen
        if n_prep == 0:
            return
        lab = torch.stack(labels_buf)
        emb = _embed_from_store(net, store, n_prep)
        logits = net.classify(emb)
        loss = criterion(logits, lab)
        tot_loss += loss.item() * n_prep
        n_seen += n_prep
        prob = torch.softmax(logits, -1)
        y_true.extend(lab.tolist())
        y_pred.extend(prob.argmax(-1).tolist())
        y_prob.extend(prob.cpu().numpy().tolist())
        store = _new_store(len(net.spec.sizes))
        labels_buf = []
        n_prep = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        for k in range(imgs.shape[0]):
            if mode == 'topm':
                coords = net.ips(imgs[k])
                px_explore += explore_px_img
            elif mode == 'scale_matched':
                ref = net.ips(imgs[k])
                px_explore += explore_px_img
                coords = net.sample_baseline('scale_matched', ref)
            elif mode == 'uniform':
                coords = net.sample_baseline('uniform')
            else:
                raise ValueError(mode)

            sids = coords[:, 0].cpu().numpy()
            scale_counts += np.bincount(sids, minlength=len(net.spec.sizes))
            px_classify += int(sum(net.spec.sizes[s] ** 2 for s in sids))
            n_img += 1

            _fill_store(net, store, imgs[k], coords, n_prep)
            labels_buf.append(labels[k])
            n_prep += 1
            if n_prep == conf.B:
                flush()
    flush()

    met = compute_metrics(y_true, y_pred, y_prob, conf.task_type)
    met['loss'] = tot_loss / max(n_seen, 1)
    cost = {'px_explore_per_img': px_explore / max(n_img, 1),
            'px_classify_per_img': px_classify / max(n_img, 1),
            'px_total_per_img': (px_explore + px_classify) / max(n_img, 1),
            'n_images': n_img,
            'scale_counts': scale_counts.tolist()}
    return met, cost


def selection_study(net, loader, conf, device, criterion):
    """Les trois strategies cote a cote : qualite et cout."""
    print("\n" + "=" * 78)
    print(f"STRATEGIES DE SELECTION — {conf.run_name}")
    print("=" * 78)
    key = conf.main_metric
    rows = {}
    for mode in ('uniform', 'scale_matched', 'topm'):
        met, cost = evaluate_selection(net, loader, conf, device, mode, criterion)
        rows[mode] = (met, cost)
        tot = sum(cost['scale_counts'])
        mix = ', '.join(f"{p}px {c}/{tot}" for p, c
                        in zip(net.spec.sizes, cost['scale_counts']))
        print(f"  {mode:14s} {key}={met.get(key, met['accuracy']):.4f}  "
              f"acc={met['accuracy']:.4f}  "
              f"pixels/img {cost['px_total_per_img']/1e6:5.2f} M  "
              f"({mix})")
    base = rows['uniform'][0].get(key, rows['uniform'][0]['accuracy'])
    sm   = rows['scale_matched'][0].get(key, rows['scale_matched'][0]['accuracy'])
    top  = rows['topm'][0].get(key, rows['topm'][0]['accuracy'])
    print(f"\n  choix des echelles  : {base:.4f} -> {sm:.4f}  "
          f"({sm - base:+.4f})")
    print(f"  choix des positions : {sm:.4f} -> {top:.4f}  "
          f"({top - sm:+.4f})")
    surcout = (rows['topm'][1]['px_total_per_img']
               / max(rows['uniform'][1]['px_total_per_img'], 1))
    print(f"  surcout de l'exploration : x{surcout:.1f} pixels")
    return rows

In [ ]:
# ============================================================
# Runner
# ============================================================
def run(key, n_epoch=None, export_saliency=True, full_scale_stats=True,
        selection_baselines=True):
    conf = CONFIGS[key]
    if n_epoch is not None:
        conf.n_epoch = n_epoch
    set_seed(conf.seed)
    print(f"\n{'='*70}\n{conf.run_name}  ({conf.model}, {conf.dataset})\n{'='*70}")

    train_loader, val_loader = build_loaders(conf)
    net = build_model(conf, device)
    px, n_cand = cost_report(conf, net)

    criterion = IPSLoss(conf.eps)
    optimizer = torch.optim.AdamW(net.parameters(), lr=0, weight_decay=conf.wd)
    is_base = conf.model == 'baseline'

    tracker = None
    try:
        from codecarbon import EmissionsTracker
        _co2_dir = OUT_HIST / 'codecarbon' / conf.run_name
        _co2_dir.mkdir(parents=True, exist_ok=True)
        tracker = EmissionsTracker(output_dir=str(_co2_dir), log_level='error')
        tracker.start()
    except Exception as exc:
        print(f"  [info] codecarbon indisponible ({exc})")

    history, best_val, best_state = [], -float('inf'), None
    t_train_tot, t_val_tot, vram_train = 0.0, 0.0, 0.0
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for ep in range(conf.n_epoch):
        t0 = time.time()
        tr_loss, ytr, ptr, qtr = run_epoch(
            net, train_loader, conf, device, optimizer, ep, criterion, is_base)
        t_train_tot += time.time() - t0
        vram_train = (torch.cuda.max_memory_allocated() / 1e6
                      if torch.cuda.is_available() else 0.0)

        t0 = time.time()
        va_loss, yva, pva, qva = run_epoch(
            net, val_loader, conf, device, None, ep, criterion, is_base)
        t_val_tot += time.time() - t0

        tr_m = compute_metrics(ytr, ptr, qtr, conf.task_type)
        va_m = compute_metrics(yva, pva, qva, conf.task_type)
        tr_m['loss'], va_m['loss'] = tr_loss, va_loss
        history.append({'epoch': ep + 1, 'train': tr_m, 'val': va_m})

        key_m = conf.main_metric
        score = va_m.get(key_m, va_m['accuracy'])
        # L'AUC vaut NaN si un lot de validation n'a qu'une classe. NaN > x
        # est faux, donc sans ce repli aucune epoque ne serait retenue et le
        # run finirait sans poids sauvegardes.
        if isinstance(score, float) and math.isnan(score):
            score = va_m['accuracy']
            key_m = key_m + '~acc'
        flag = ''
        if score > best_val:
            best_val, flag = score, '  *'
            best_state = {k: v.detach().cpu().clone()
                          for k, v in net.state_dict().items()}
        print(f"  ep {ep+1:3d}/{conf.n_epoch}  "
              f"train {tr_loss:.4f}  val {va_loss:.4f}  "
              f"{key_m} {score:.4f}{flag}")

    co2 = 0.0
    if tracker is not None:
        try:
            co2 = float(tracker.stop() or 0.0)
        except Exception:
            co2 = 0.0

    timing = {
        'train_vram_mb': round(vram_train, 1),
        'train_time_s': round(t_train_tot, 1),
        'val_time_s': round(t_val_tot, 1),
        'total_co2_kg': co2,
        'co2_per_epoch_kg': co2 / max(conf.n_epoch, 1),
    }
    if best_state is not None:
        torch.save(best_state, OUT_CKPT / f"{conf.run_name}_weights.pt")
        net.load_state_dict(best_state)
    with open(OUT_HIST / f"{conf.run_name}_history.json", 'w') as f:
        json.dump(history, f, indent=2)
    log_experiment(OUT_EXPE, conf, net, history, timing, px, n_cand)

    # Analyses d'apres-entrainement. Poids, historique et CSV sont deja
    # ecrits : si l'une echoue, on perd l'analyse, pas le run.
    if not is_base:
        if (full_scale_stats and conf.model == 'selector'
                and len(net.spec.sizes) > 1):
            try:
                scale_distribution(net, val_loader, device)
            except Exception as exc:
                print(f"  [echec] repartition des echelles : {exc!r}")
        if selection_baselines:
            try:
                rows = selection_study(net, val_loader, conf, device, criterion)
                with open(OUT_EXPE / f"{conf.run_name}_selection.json", 'w') as f:
                    json.dump({m: {'metrics': v[0], 'cost': v[1]}
                               for m, v in rows.items()}, f, indent=2)
            except Exception as exc:
                print(f"  [echec] strategies de selection : {exc!r}")
        if export_saliency:
            try:
                save_saliency(net, val_loader.dataset, conf,
                              OUT_SAL / conf.dataset, n_images=5,
                              seed=conf.seed)
            except Exception as exc:
                print(f"  [echec] saillance : {exc!r}")
    print(f"  meilleur {conf.main_metric} = {best_val:.6f}")
    return history, timing


# ------------------------------------------------------------------
# Selection des runs.
#   IPS_RUNS   : liste separee par des virgules, sinon tout
#   IPS_EPOCHS : force le nombre d'epoques (utile pour un essai a 1)
# ------------------------------------------------------------------
ALL_RUNS = [
    'ddr_baseline', 'ddr_ips', 'ddr_msips2', 'ddr_msips3',
    'ddr_hmsips_d2', 'ddr_hmsips_d4', 'ddr_hmsips_d8', 'ddr_hmsips_d16',
    'ddr_hmsips3',
    'ddr_msips3_clin', 'ddr_ips_b2', 'ddr_msips2_b2',
    'rsna_baseline', 'rsna_ips', 'rsna_msips2', 'rsna_msips3',
    'rsna_msips2_clin',
    'rsna_hmsips_d2', 'rsna_hmsips_d4', 'rsna_hmsips_d8', 'rsna_hmsips3',
]

_sel = os.environ.get('IPS_RUNS', '').strip()
TO_RUN = [k.strip() for k in _sel.split(',') if k.strip()] if _sel else ALL_RUNS
_ep = os.environ.get('IPS_EPOCHS', '').strip()
EPOCHS = int(_ep) if _ep else None

unknown = [k for k in TO_RUN if k not in CONFIGS]
if unknown:
    raise ValueError(f"runs inconnus : {unknown}. Disponibles : {list(CONFIGS)}")

print(f"A lancer : {TO_RUN}" + (f"  ({EPOCHS} epoques)" if EPOCHS else ""))

_failed = []
for _key in TO_RUN:
    try:
        run(_key, n_epoch=EPOCHS)
    except Exception as _exc:
        import traceback
        _failed.append((_key, repr(_exc)))
        print(f"\n[ECHEC] {_key} : {_exc!r}")
        traceback.print_exc()
        print("  -> on continue avec les runs suivants\n")

print("\n" + "=" * 70)
if _failed:
    print(f"{len(_failed)} run(s) en echec sur {len(TO_RUN)} :")
    for _k, _e in _failed:
        print(f"  {_k} : {_e}")
else:
    print(f"Les {len(TO_RUN)} run(s) se sont termines.")
print("=" * 70)

A lancer : ['ddr_ips', 'ddr_hmsips_d2']  (1 epoques)

ddr_ips_v2  (selector, ddr)
[DDR] train |  6260 images | C0=3133, C1=315, C2=2238, C3=118, C4=456
[DDR] valid |  2503 images | C0=1253, C1=126, C2=895, C3=47, C4=182
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 151MB/s] 


  candidats          : 49
  pixels encodes     : 0.80 M  (1.00 x H*W)
  cartes             : 128px->4x4
  pic patchs (M+I)   : 3.9 Mo + image 9.6 Mo
  [info] codecarbon indisponible (No module named 'codecarbon')
  ep   1/1  train 1.2177  val 0.9054  kappa 0.5136  *
  CSV : /kaggle/working/experiments/ddr_ips_v2_experiment.csv

STRATEGIES DE SELECTION — ddr_ips_v2
  uniform        kappa=0.5305  acc=0.6580  pixels/img  0.16 M  (128px 25030/25030)
  scale_matched  kappa=0.5121  acc=0.6472  pixels/img  0.97 M  (128px 25030/25030)
  topm           kappa=0.5136  acc=0.6672  pixels/img  0.97 M  (128px 25030/25030)

  choix des echelles  : 0.5305 -> 0.5121  (-0.0185)
  choix des positions : 0.5121 -> 0.5136  (+0.0015)
  surcout de l'exploration : x5.9 pixels
  repartition des echelles sur ces images : 128px 50/50
  meilleur kappa = 0.513559

ddr_hmsips_d2_v2  (hierarchical, ddr)
[DDR] train |  6260 images | C0=3133, C1=315, C2=2238, C3=118, C4=456
[DDR] valid |  2503 images | C0=1253, C1=126,